# 02 - Geocoding & MRT Distance Feature Engineering
ST1516 - DevOps & Applied Analytics | HDB Resale Price Prediction Web Application | Branch: geocoding  
Author: <your name> | Date: <run date>

- Load cleaned HDB resale data and build a unique address registry for geocoding
- Call OneMap with cached, token-based requests and run primary + fallback passes
- Enrich with OSM amenities and compute distance-to-amenity features
- Export geo and modelling datasets: hdb_with_geo.csv, hdb_geo_model.csv, hdb_geo_amen_model.csv



## 1. Load Phase 01 cleaned dataset
Load `../data/intermediate/hdb_clean.csv` from notebook 01; this ~180k-row table is the canonical starting point for all downstream geospatial features.



In [ ]:
import pandas as pd
import numpy as np
import json
import requests
from math import radians, sin, cos, sqrt, atan2

clean_path = "../data/intermediate/hdb_clean.csv"
df = pd.read_csv(clean_path)

df.shape, df.head()


## 2. Construct unique address registry
Build `address_key` from `block`, `street_name`, and `town` plus "Singapore" to form stable geocoding queries; deduping prevents ~180k repeated API calls and helps stay within OneMap rate limits.



In [ ]:
df["address_key"] = (
    df["block"].astype(str).str.strip() + " " +
    df["street_name"].str.strip() + ", " +
    df["town"].str.strip() + ", Singapore"
)

addr_df = df[["address_key"]].drop_duplicates().reset_index(drop=True)
addr_df.shape, addr_df.head()


## 3. OneMap auth + session + cache initialisation
Load token-based OneMap credentials from `.env`, create a session, and back all lookups with `../data/cache/onemap_cache.json` so reruns are crash-safe and reuse prior results.



In [ ]:
import os
import json
import time
import requests
from tqdm import tqdm
from dotenv import load_dotenv

load_dotenv()

# =========================================
# 1. Get OneMap Token (AUTHENTICATED)
# =========================================
def get_onemap_token():
    url = "https://www.onemap.gov.sg/api/auth/post/getToken"
    email = os.getenv("ONEMAP_EMAIL")
    password = os.getenv("ONEMAP_EMAIL_PASSWORD")

    if not email or not password:
        raise RuntimeError("Missing OneMap credentials in .env")

    payload = {"email": email, "password": password}
    resp = requests.post(url, json=payload, timeout=10)
    resp.raise_for_status()
    data = resp.json()
    print("Token expiry:", data["expiry_timestamp"])
    return data["access_token"]

ACCESS_TOKEN = get_onemap_token()

# =========================================
# 2. Authenticated Elastic Search Endpoint
# =========================================
BASE_URL = "https://www.onemap.gov.sg/api/common/elastic/search"
session = requests.Session()
session.headers.update({"Authorization": ACCESS_TOKEN})

# =========================================
# 3. Load / Init Cache
# =========================================
CACHE_FILE = "../data/cache/onemap_cache.json"
os.makedirs("../data/cache", exist_ok=True)

try:
    with open(CACHE_FILE, "r", encoding="utf-8") as f:
        cache = json.load(f)
    print("Loaded cache entries:", len(cache))
except FileNotFoundError:
    cache = {}
    print("No existing cache, starting fresh.")


Cache helpers to generate cache keys and read/write the OneMap JSON cache supporting the geocoding session.



In [ ]:
def cache_key(block, street, town):
    b = str(block).strip().upper()
    s = str(street).strip().upper()
    t = str(town).strip().upper()
    return f"{b}|{s}|{t}"

def geocode_one(block, street, town):
    if pd.isna(block) or pd.isna(street) or pd.isna(town):
        return None, None

    key = cache_key(block, street, town)

    # Cache hit
    if key in cache:
        entry = cache[key]
        return entry["lat"], entry["lon"]

    query = f"{block} {street} {town} Singapore"
    params = {
        "searchVal": query,
        "returnGeom": "Y",
        "getAddrDetails": "Y",
        "pageNum": 1,
    }

    try:
        r = session.get(BASE_URL, params=params, timeout=8)
        r.raise_for_status()
        data = r.json()
        results = data.get("results", [])

        if not results:
            lat = lon = None
        else:
            best = results[0]
            lat = float(best["LATITUDE"])
            lon = float(best["LONGITUDE"])

        cache[key] = {"lat": lat, "lon": lon}
        return lat, lon

    except Exception as e:
        print(f"[WARN] Failed for {query}: {e}")
        cache[key] = {"lat": None, "lon": None}
        return None, None


## 4. Primary geocoding pass
Loop over unique addresses with "<block> <street> <town> Singapore" queries, throttling between requests and caching responses; some addresses remain unresolved after this first pass.



In [ ]:
addr_cols = ["block", "street_name", "town"]

unique_addrs = (
    df[addr_cols]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Unique addresses:", len(unique_addrs))

lats = []
lons = []

for idx, row in tqdm(unique_addrs.iterrows(), total=len(unique_addrs)):
    lat, lon = geocode_one(
        row["block"],
        row["street_name"],
        row["town"]
    )

    lats.append(lat)
    lons.append(lon)

    # SAVE CACHE EVERY 20 REQUESTS (CRASH SAFE)
    if idx % 20 == 0:
        with open(CACHE_FILE, "w", encoding="utf-8") as f:
            json.dump(cache, f, indent=2)

    time.sleep(0.15)

unique_addrs["latitude"] = lats
unique_addrs["longitude"] = lons

# Final cache save
with open(CACHE_FILE, "w", encoding="utf-8") as f:
    json.dump(cache, f, indent=2)

print(unique_addrs[["latitude", "longitude"]].isna().sum())


## 5. Failure analysis (suffix/token/town profiling)
Profile unresolved addresses by street suffix, token, and town to pinpoint patterns; failures cluster in specific estates, suggesting OneMap coverage gaps rather than malformed inputs.



## 6. Failure analysis (suffix/token/town profiling)
Profile unresolved addresses by suffix, uppercase tokens, and town to see if formatting drives misses; failures cluster in specific estates and towns, pointing to OneMap coverage gaps more than bad input.

Check suffix distributions to confirm common road endings appear in both successful and failed lookups.



In [ ]:
suffix_series = df["street_name"].str.extract(r"\b(AVE|AVENUE|ST|STREET|RD|ROAD|DR|CRES|LOOP|HWAY|WAY)\b", expand=False)
suffix_series.value_counts(dropna=False)


Check high-frequency uppercase tokens to verify street text integrity across the dataset.



In [ ]:
token_counts = (
    df["street_name"]
    .str.findall(r"\b[A-Z]{2,}\b")
    .explode()
    .value_counts()
)

token_counts.head(30)


Inspect unresolved addresses to pinpoint repeated streets and towns driving most misses.



In [ ]:
failed = unique_addrs[unique_addrs["latitude"].isna()]
failed["street_name"].value_counts().head(30)


Top failing towns highlight estates likely affected by OneMap coverage gaps rather than bad input formatting.



In [ ]:
failed["town"].value_counts().head(20)


In [ ]:
success = unique_addrs[unique_addrs["latitude"].notna()]

print("FAILED suffixes:")
print(failed["street_name"].str.extract(r"\b([A-Z]{2,})\b")[0].value_counts().head(10))

print("\nSUCCESS suffixes:")
print(success["street_name"].str.extract(r"\b([A-Z]{2,})\b")[0].value_counts().head(10))


## 6. Multi-pass fallback geocoding
Progressively relax queries (block+street+town -> block+street -> street+town -> street) to recover remaining points; coverage rises to ~99% with only a small residue unresolved.



In [ ]:
failed_addrs = unique_addrs[unique_addrs["latitude"].isna()].copy()

def geocode_fallback(block, street, town):
    queries = [
        f"{block} {street} {town} Singapore",
        f"{block} {street} Singapore",
        f"{street} {town} Singapore",
        f"{street} Singapore"
    ]

    for q in queries:
        try:
            params = {
                "searchVal": q,
                "returnGeom": "Y",
                "getAddrDetails": "Y",
                "pageNum": 1,
            }
            r = session.get(BASE_URL, params=params, timeout=8)
            r.raise_for_status()
            data = r.json()
            results = data.get("results", [])

            if results:
                best = results[0]
                return float(best["LATITUDE"]), float(best["LONGITUDE"])
        except:
            continue

    return None, None


In [ ]:
for idx, row in tqdm(failed_addrs.iterrows(), total=len(failed_addrs)):
    lat, lon = geocode_fallback(
        row["block"],
        row["street_name"],
        row["town"]
    )

    failed_addrs.at[idx, "latitude"] = lat
    failed_addrs.at[idx, "longitude"] = lon


In [ ]:
unique_addrs.update(failed_addrs)


In [ ]:
unique_addrs[["latitude", "longitude"]].isna().sum()


Fallback results: success rate improves to about 99%, leaving a small residual set likely outside OneMap coverage.



## 7. Merge coordinates and persist geo-enriched dataset
Merge latitude/longitude back to the full dataset on block, street_name, and town, and write `hdb_with_geo.csv` retaining all rows, including those still missing coordinates.



In [ ]:
# 9. Merge geocodes into full cleaned dataset

# Keep only the necessary columns from unique_addrs
geo_cols = ["block", "street_name", "town", "latitude", "longitude"]
unique_addrs_geo = unique_addrs[geo_cols].copy()

# Merge back on block, street_name, town
df_geo = df.merge(
    unique_addrs_geo,
    on=["block", "street_name", "town"],
    how="left"
)

print("Geo-enriched shape:", df_geo.shape)
df_geo[["latitude", "longitude"]].isna().sum()


In [ ]:
# Persist full geo-enriched dataset for reuse
geo_output_path = "../data/intermediate/hdb_with_geo.csv"
df_geo.to_csv(geo_output_path, index=False)
geo_output_path


## 8. Geo-model subset
Drop rows without coordinates to form `hdb_geo_model.csv`; use this trimmed table whenever distance-based features are required.



In [ ]:
# 10. Create subset with valid coordinates only

df_geo_model = df_geo.dropna(subset=["latitude", "longitude"]).copy()
print("Geo-model subset shape:", df_geo_model.shape)

geo_model_output_path = "../data/intermediate/hdb_geo_model.csv"
df_geo_model.to_csv(geo_model_output_path, index=False)
geo_model_output_path


In [ ]:
df_geo = pd.read_csv("../data/intermediate/hdb_with_geo.csv")


In [ ]:
import folium

# Center of SG from your data
center_lat = df_geo["latitude"].mean()
center_lon = df_geo["longitude"].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=11)

# Don't plot all 180k points, only a small sample
points = df_geo.dropna(subset=["latitude", "longitude"]).sample(3000, random_state=42)

for _, row in points.iterrows():
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=2,
        weight=0,
        fill=True,
        fill_opacity=0.5
    ).add_to(m)

m


## 9. OSM amenity extraction
Build a study-area polygon from HDB points and query OSM for transport, retail, education, health, and lifestyle amenities; standardise amenity types and ensure point geometries for downstream distance calculations.



In [ ]:
import osmnx as ox
from shapely.geometry import Point

# 1) Get a bounding polygon from your HDB points
from shapely.geometry import MultiPoint

pts = MultiPoint([
    Point(xy) for xy in zip(
        df_geo["longitude"].dropna(),
        df_geo["latitude"].dropna()
    )
])
study_area = pts.convex_hull.buffer(0.01)  # small buffer around HDB cluster

# 2) Pull amenities from OSM within this polygon
tags = {
    # Transport
    "railway": ["station"],
    "highway": ["bus_stop"],

    # Education & health
    "amenity": [
        "school",
        "clinic",
        "hospital",
        "kindergarten",
        "restaurant",
        "food_court"
    ],

    # Retail
    "shop": [
        "supermarket",
        "mall"
    ],

    # Lifestyle
    "leisure": [
        "fitness_centre",
        "park"
    ]
}

gdf_amen = ox.features_from_polygon(study_area, tags=tags)

gdf_amen.head()


In [ ]:
gdf_amen_clean = gdf_amen[
    ["amenity", "shop", "leisure", "railway", "highway", "name", "geometry"]
].copy()

# Collapse all tag columns into ONE amenity_type
gdf_amen_clean["amenity_type"] = (
    gdf_amen_clean["amenity"]
    .fillna(gdf_amen_clean["shop"])
    .fillna(gdf_amen_clean["leisure"])
    .fillna(gdf_amen_clean["railway"])
    .fillna(gdf_amen_clean["highway"])
)

# Keep only rows that actually became a type
gdf_amen_clean = gdf_amen_clean[gdf_amen_clean["amenity_type"].notna()].copy()

# Force all geometries to points
gdf_amen_clean["geometry"] = gdf_amen_clean["geometry"].apply(
    lambda g: g if g.geom_type == "Point" else g.centroid
)

gdf_amen_clean["lon"] = gdf_amen_clean.geometry.x
gdf_amen_clean["lat"] = gdf_amen_clean.geometry.y

amen_coords = gdf_amen_clean[
    ["amenity_type", "name", "lat", "lon"]
].dropna()

amen_coords["amenity_type"].value_counts()


In [ ]:
import folium

center_lat = df_geo["latitude"].mean()
center_lon = df_geo["longitude"].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=11)


## 10. Build BallTrees and compute distance features
Create haversine BallTrees per amenity layer and compute nearest distances (km) for each flat. Distance columns:
- dist_mrt_km
- dist_school_km
- dist_supermarket_km
- dist_health_km



In [ ]:
# 11. PREPARE AMENITY DISTANCE FEATURES FOR MODELLING

from sklearn.neighbors import BallTree
import numpy as np

# Work ONLY on rows with valid coordinates
df_model = pd.read_csv("../data/intermediate/hdb_geo_model.csv")
print("Modelling base shape:", df_model.shape)

# HDB coordinates in radians
hdb_coords = np.radians(
    df_model[["latitude", "longitude"]].values
)


amen_mrt = amen_coords[amen_coords["amenity_type"] == "station"].copy()
amen_school = amen_coords[amen_coords["amenity_type"] == "school"].copy()
amen_supermarket = amen_coords[amen_coords["amenity_type"] == "supermarket"].copy()
amen_health = amen_coords[
    amen_coords["amenity_type"].isin(["clinic", "hospital"])
].copy()

print("\nCounts used for modelling:")
print("MRT:", len(amen_mrt))
print("School:", len(amen_school))
print("Supermarket:", len(amen_supermarket))
print("Health (clinic + hospital):", len(amen_health))

# ---- 11.2 Build BallTrees (skip if a category is empty) ----

def build_tree(df, label):
    if df.empty:
        print(f"[WARN] No amenities found for '{label}', skipping this category.")
        return None
    coords = np.radians(df[["lat", "lon"]].values)
    return BallTree(coords, metric="haversine")

trees = {
    "mrt": build_tree(amen_mrt, "mrt"),
    "school": build_tree(amen_school, "school"),
    "supermarket": build_tree(amen_supermarket, "supermarket"),
    "health": build_tree(amen_health, "health"),
}

EARTH_RADIUS_KM = 6371.0


In [ ]:
# 12. COMPUTE NEAREST DISTANCES (KM) FOR EACH AMENITY TYPE

def nearest_distance_km(tree, label):
    """
    Given a BallTree and HDB coords, return the nearest distance in km
    for each HDB row. If tree is None, returns a column of NaNs.
    """
    if tree is None:
        print(f"[WARN] No tree for '{label}', filling with NaN.")
        return np.full(shape=(len(df_model),), fill_value=np.nan)

    # BallTree.haversine returns distance in radians
    dist_rad, _ = tree.query(hdb_coords, k=1)
    return dist_rad[:, 0] * EARTH_RADIUS_KM

df_model["dist_mrt_km"] = nearest_distance_km(trees["mrt"], "mrt")
df_model["dist_school_km"] = nearest_distance_km(trees["school"], "school")
df_model["dist_supermarket_km"] = nearest_distance_km(trees["supermarket"], "supermarket")
df_model["dist_health_km"] = nearest_distance_km(trees["health"], "health")

df_model[[
    "dist_mrt_km",
    "dist_school_km",
    "dist_supermarket_km",
    "dist_health_km"
]].describe()


## 11. Save final modelling dataset
Export `hdb_geo_amen_model.csv` containing structural, geo, and distance features; this is the main input for the modelling notebook.



In [ ]:
# 13. SAVE FINAL GEO + AMENITY MODELLING DATASET

final_model_path = "../data/intermediate/hdb_geo_amen_model.csv"
df_model.to_csv(final_model_path, index=False)
final_model_path


## 12. Quick EDA on distance features
Histograms show right-skewed distances with most flats within about 1 km of key amenities. Town-level summaries highlight which estates sit closer or farther on average to MRT and other services. Correlations show weak negative links between resale price and distance to MRT/supermarkets/health facilities. Quartiles and scatterplots indicate flats nearer to MRT/amenities tend to be pricier on average, though other factors still influence prices.



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

dist_cols = [
    "dist_mrt_km",
    "dist_school_km",
    "dist_supermarket_km",
    "dist_health_km",
]

plt.figure(figsize=(12, 8))

for i, col in enumerate(dist_cols, 1):
    plt.subplot(2, 2, i)
    sns.histplot(df_model[col], bins=40, kde=True)
    plt.title(f"Distribution of {col}")
    plt.xlabel("Distance (km)")
    plt.ylabel("Count")

plt.tight_layout()
plt.show()


Town-level averages highlight estates with shorter MRT/amenity distances and those located farther from key services.



In [ ]:
town_dist_summary = (
    df_model
    .groupby("town")[dist_cols]
    .mean()
    .sort_values("dist_mrt_km")   # sort by MRT proximity
)

town_dist_summary.round(3).head(10)


In [ ]:
town_dist_summary.round(3).tail(10)


Correlation matrix shows weak negative relationships between price and distances to MRT, supermarkets, and health facilities.



In [ ]:
corr_cols = ["resale_price"] + dist_cols
df_model[corr_cols].corr().round(3)


Quartile summaries: flats nearer to MRT or amenities tend to command higher average prices, though other factors still matter.



In [ ]:
def price_by_distance_bin(col, q=4):
    label = col + "_bin"
    df_tmp = df_model[[col, "resale_price"]].copy()
    df_tmp[label] = pd.qcut(df_tmp[col], q=q, labels=[f"Q{i}" for i in range(1, q+1)])

    return (
        df_tmp
        .groupby(label)["resale_price"]
        .agg(["count", "mean", "median"])
        .round(0)
    )

price_by_distance_bin("dist_mrt_km")


In [ ]:
price_by_distance_bin("dist_supermarket_km")


In [ ]:
price_by_distance_bin("dist_school_km")


In [ ]:
price_by_distance_bin("dist_health_km")


Scatterplots reinforce the general downward trend between price and distance, with substantial spread driven by other attributes.



In [ ]:
plt.figure(figsize=(12, 4))

for i, col in enumerate(["dist_mrt_km", "dist_supermarket_km"], 1):
    plt.subplot(1, 2, i)
    sns.scatterplot(
        data=df_model.sample(8000, random_state=42),
        x=col,
        y="resale_price",
        alpha=0.2
    )
    sns.regplot(
        data=df_model.sample(8000, random_state=42),
        x=col,
        y="resale_price",
        scatter=False,
        color="red"
    )
    plt.title(f"Resale price vs {col}")
    plt.xlabel("Distance (km)")
    plt.ylabel("Resale price")

plt.tight_layout()
plt.show()


## 13. Final summary
- Geocoding coverage reaches about 99% after primary and fallback passes
- Persisted outputs: hdb_with_geo.csv (all rows), hdb_geo_model.csv (geo-only rows), hdb_geo_amen_model.csv (geo + distance features)
- Distance features capture accessibility to MRT, schools, supermarkets, and health facilities for modelling
- Pipeline is cached, tokenised, and rerunnable without repeating resolved API calls

